In [4]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [5]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [6]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

In [7]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

dataset

In [9]:
messages = []
prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing a task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "```json")
text = chat(messages, stop_sequences=["```"])
print("Raw text:", repr(text))
dataset = json.loads(text)
print("Dataset:", dataset)

Raw text: '\n[\n    {\n        "task": "Write a Python function that parses an AWS CloudFormation template (JSON string) and extracts all resource logical IDs"\n    },\n    {\n        "task": "Create a regular expression that matches valid AWS S3 bucket names (3-63 characters, lowercase letters, numbers, and hyphens, must start and end with alphanumeric)"\n    },\n    {\n        "task": "Write a Python function that takes an AWS IAM policy JSON object and returns a list of all resource ARNs referenced in the policy"\n    }\n]\n'
Dataset: [{'task': 'Write a Python function that parses an AWS CloudFormation template (JSON string) and extracts all resource logical IDs'}, {'task': 'Create a regular expression that matches valid AWS S3 bucket names (3-63 characters, lowercase letters, numbers, and hyphens, must start and end with alphanumeric)'}, {'task': 'Write a Python function that takes an AWS IAM policy JSON object and returns a list of all resource ARNs referenced in the policy'}]


In [10]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

print("Saved successfully!")
print(json.dumps(dataset, indent=2))


Saved successfully!
[
  {
    "task": "Write a Python function that parses an AWS CloudFormation template (JSON string) and extracts all resource logical IDs"
  },
  {
    "task": "Create a regular expression that matches valid AWS S3 bucket names (3-63 characters, lowercase letters, numbers, and hyphens, must start and end with alphanumeric)"
  },
  {
    "task": "Write a Python function that takes an AWS IAM policy JSON object and returns a list of all resource ARNs referenced in the policy"
  }
]


In [11]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    # TODO - Grading
    score = 10
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [12]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS CloudFormation Template Parser\n\nHere's a Python function that parses an AWS CloudFormation template and extracts all resource logical IDs:\n\n```python\nimport json\nfrom typing import List\n\ndef extract_resource_ids(template: str) -> List[str]:\n    \"\"\"\n    Parses an AWS CloudFormation template (JSON string) and extracts \n    all resource logical IDs.\n    \n    Args:\n        template (str): CloudFormation template as a JSON string\n        \n    Returns:\n        List[str]: List of resource logical IDs\n        \n    Raises:\n        json.JSONDecodeError: If the template is not valid JSON\n        KeyError: If the template doesn't contain a Resources section\n    \"\"\"\n    try:\n        template_dict = json.loads(template)\n    except json.JSONDecodeError as e:\n        raise json.JSONDecodeError(f\"Invalid JSON template: {e.msg}\", e.doc, e.pos)\n    \n    # Extract the Resources section\n    if \"Resources\" not in template_dict:\n        raise